## Data Analysis

Doris provides a rich set of window functions and analytical functions. The following simple examples demonstrate how to use typical window functions and analytical functions.

When processing observability trace data, long-lived connections and streaming services often contain extremely dense arrays of internal events. How can Doris calculate the top three longest-running spans for each `service_name` and count the number of events contained in each span?

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [3]:
lab.sql(r"""
WITH ranked_traces AS (
    SELECT
        service_name,
        trace_id,
        span_id,
        span_name,
        timestamp,
        ROUND(duration / 1000000.0, 2) AS duration_ms,
        size(events) AS event_count,
        ROW_NUMBER() OVER (
            PARTITION BY service_name
            ORDER BY duration DESC
        ) AS rank_num
    FROM
        otel_traces
    WHERE
        span_kind = 'SPAN_KIND_SERVER'
)
SELECT
    service_name,
    trace_id,
    span_id,
    span_name,
    timestamp,
    duration_ms,
    event_count,
    rank_num
FROM
    ranked_traces
WHERE
    rank_num <= 3
ORDER BY
    service_name,
    rank_num;
""", title='Longest server spans by service')


service_name,trace_id,span_id,span_name,timestamp,duration_ms,event_count,rank_num


,service_name,trace_id,span_id,span_name,timestamp,duration_ms,event_count,rank_num


First, `size(events)` counts the number of elements in the array, which is the number of message events. The query then partitions rows by `service_name` and assigns a sequential rank starting from 1 based on `duration` in descending order. The outer query uses `rank_num <= 3` to obtain the three longest-running records for each service.

This is a familiar use of window functions.

What if you need to calculate P90 and P99 response latency, maximum latency, and the average number of events per request by service and API? How can this requirement be implemented in Doris?

This requirement is a straightforward aggregate query:

In [4]:
lab.sql(r"""
SELECT
    service_name,
    span_name,
    COUNT(*) AS total_spans,

    -- 1. Basic duration analysis (convert nanoseconds to milliseconds)
    ROUND(AVG(duration) / 1000000.0, 2) AS avg_duration_ms,
    ROUND(MAX(duration) / 1000000.0, 2) AS max_duration_ms,

    -- 2. Duration percentile analysis (P90 / P99)
    ROUND(PERCENTILE_APPROX(duration, 0.90) / 1000000.0, 2) AS p90_duration_ms,
    ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000000.0, 2) AS p99_duration_ms,

    -- 3. Average event count per request
    ROUND(AVG(size(events)), 1) AS avg_event_count
FROM
    otel_traces
WHERE
    span_kind = 'SPAN_KIND_SERVER'
GROUP BY
    service_name,
    span_name
ORDER BY
    p99_duration_ms DESC;
""", title='Latency and event-count summary')


service_name,span_name,total_spans,avg_duration_ms,max_duration_ms,p90_duration_ms,p99_duration_ms,avg_event_count


,service_name,span_name,total_spans,avg_duration_ms,max_duration_ms,p90_duration_ms,p99_duration_ms,avg_event_count


As shown, window functions and aggregate functions are straightforward to use in Doris.

## Litefuse

This section explains how to start, configure, and view Litefuse step by step.

Litefuse currently offers a cloud mode, which is the fastest way to get started. Register and start using Litefuse immediately at [https://litefuse.cloud/auth/sign-in](https://litefuse.cloud/auth/sign-in).

After registering, sign in to Litefuse Cloud and select the agent data to view. Litefuse Cloud displays sample data.

<img src="images/sign_in.png" alt="Signing in to Litefuse Cloud" style="max-width:100%;height:auto;">

Select `demo-project`:

<img src="images/project.png" alt="Selecting the demo project in Litefuse Cloud" style="max-width:100%;height:auto;">

Click Dashboard:

<img src="images/litefuse-dashboard.png" alt="Litefuse Cloud dashboard" style="max-width:100%;height:auto;">

Select the time range:

<img src="images/choose_dataset.png" alt="Selecting the time range for Litefuse analysis" style="max-width:100%;height:auto;">

After completing these steps, you can perform data analysis in Litefuse.

To monitor your own agent, ingest data into Litefuse as follows.

In [ ]:
%pip install langfuse openai


In [ ]:
%env LANGFUSE_SECRET_KEY=sk-lf-...
%env LANGFUSE_PUBLIC_KEY=pk-lf-...
%env LANGFUSE_BASE_URL=https://litefuse.cloud


In [ ]:
from langfuse import observe
from langfuse.openai import openai  # OpenAI integration

@observe()
def story():
    return openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "What is Litefuse?"}],
    ).choices[0].message.content

@observe()
def main():
    return story()

main()
